# Meta-Review Evaluation — Google Colab
依次运行每个 Cell，每段结束后可以看到该指标的结果。

## Cell 1：安装依赖

In [ ]:
!pip install -q transformers rouge_score bert-score sentence-transformers scikit-learn tqdm

## Cell 2：挂载 Google Drive & 设置路径

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/meta_review')

# evaluation.py 在 src/eval/ 里
sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'eval'))

# ── 修改这里来切换评估哪个模型 ──────────────────────────────
PRED_FILE = 'data/predictions/flan_t5_run1_predictions.csv'
TEST_FILE = 'data/processed/test.jsonl'
# ──────────────────────────────────────────────────────────

PRED_PATH = PROJECT_ROOT / PRED_FILE
TEST_PATH = PROJECT_ROOT / TEST_FILE
OUT_PATH  = PRED_PATH.parent / f'{PRED_PATH.stem}_eval_results.json'

assert PRED_PATH.exists(), f'找不到预测文件: {PRED_PATH}'
assert TEST_PATH.exists(), f'找不到测试集: {TEST_PATH}'
print(f'✅ 预测文件: {PRED_PATH}')
print(f'✅ 测试集:   {TEST_PATH}')
print(f'✅ 结果将保存到: {OUT_PATH}')

## Cell 3：加载数据

In [ ]:
import csv, json, re
from typing import List, Dict
import numpy as np

# 加载预测 CSV
with open(PRED_PATH, 'r', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

gold_decisions = [r['true_decision']    for r in rows]
pred_decisions = [r['pred_decision']    for r in rows]
gold_reviews   = [r['true_meta_review'] for r in rows]
pred_reviews   = [r['pred_meta_review'] for r in rows]
paper_ids      = [r['paper_id']         for r in rows]

print(f'✅ 加载了 {len(rows)} 条预测样本')
print(f'\n第一条样本预览：')
print(f'  paper_id:      {paper_ids[0]}')
print(f'  true_decision: {gold_decisions[0]}')
print(f'  pred_decision: {pred_decisions[0]}')
print(f'  pred_review 前100字: {pred_reviews[0][:100]}')

## Cell 4：从 test.jsonl 提取原始 Reviews

In [ ]:
def parse_reviews_from_input(input_text: str) -> List[str]:
    """从 input_text 中按 REVIEW N: 分割，提取每条 review 文本。"""
    match = re.search(r'\nREVIEWS:\n', input_text)
    if not match:
        return [input_text]
    reviews_block = input_text[match.end():]
    parts = re.split(r'\nREVIEW \d+:\n', reviews_block)
    reviews = [p.strip() for p in parts if p.strip()]
    return reviews if reviews else [reviews_block]


reviews_per_paper: Dict[str, List[str]] = {}
with open(TEST_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ex = json.loads(line)
        reviews_per_paper[ex['paper_id']] = parse_reviews_from_input(ex['input_text'])

print(f'✅ 加载了 {len(reviews_per_paper)} 篇论文的原始 reviews')

# 预览第一篇
pid   = paper_ids[0]
revs  = reviews_per_paper[pid]
print(f'\n论文 {pid} 共有 {len(revs)} 条 review')
print(f'第一条 review 前150字:\n{revs[0][:150]}')

## Cell 5：决策分类指标（Accuracy / F1）

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def to_binary(lbl: str) -> int:
    return 1 if str(lbl).strip().upper().startswith('ACCEPT') else 0

y_true = [to_binary(d) for d in gold_decisions]
y_pred = [to_binary(d) for d in pred_decisions]

unknown_count = sum(
    1 for p in pred_decisions
    if not str(p).strip().upper().startswith(('ACCEPT', 'REJECT'))
)

decision_metrics = {
    'accuracy':        float(accuracy_score(y_true, y_pred)),
    'precision_macro': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
    'recall_macro':    float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
    'f1_macro':        float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    'unknown_count':   unknown_count,
    'total_samples':   len(rows),
}

print('📊 决策分类指标：')
for k, v in decision_metrics.items():
    print(f'  {k}: {v}')

## Cell 6：ROUGE（文本重叠质量）

In [ ]:
from tqdm import tqdm
from evaluation import compute_rouge_all

r1, r2, rl = [], [], []
for gen, gold in tqdm(zip(pred_reviews, gold_reviews), total=len(rows), desc='ROUGE'):
    s = compute_rouge_all(gen, gold)
    r1.append(s['rouge1'])
    r2.append(s['rouge2'])
    rl.append(s['rougeL'])

rouge_metrics = {
    'rouge1':    float(np.mean(r1)),
    'rouge2':    float(np.mean(r2)),
    'rougeL':    float(np.mean(rl)),
    'rougeLsum': float(np.mean(rl)),
}

print('📊 ROUGE 指标（越大越好，0~1）：')
for k, v in rouge_metrics.items():
    print(f'  {k}: {v:.4f}')

## Cell 7：BERTScore（语义相似度）

In [ ]:
from bert_score import score as bert_score_fn

print('⏳ 计算 BERTScore（需要下载模型，请稍候）...')
P, R, F1 = bert_score_fn(pred_reviews, gold_reviews, lang='en', verbose=False)

bertscore_metrics = {
    'bertscore_precision': float(P.mean()),
    'bertscore_recall':    float(R.mean()),
    'bertscore_f1':        float(F1.mean()),
}

print('📊 BERTScore（越大越好，通常在 0.85~0.95）：')
for k, v in bertscore_metrics.items():
    print(f'  {k}: {v:.4f}')

## Cell 8：Coverage（所有句子）& Hallucination（最慢，可跳过）

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoModelForSequenceClassification, AutoTokenizer as HFTokenizer
from evaluation import hallucination_rate

THRESHOLD = 0.80
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('⏳ 加载 SentenceTransformer...')
emb_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print(f'⏳ 加载 NLI 模型 (facebook/bart-large-mnli) on {device}...')
nli_tokenizer = HFTokenizer.from_pretrained('facebook/bart-large-mnli')
nli_model     = AutoModelForSequenceClassification.from_pretrained(
    'facebook/bart-large-mnli'
).to(device)
nli_model.eval()

def coverage_all_sentences(generated: str, gold: str, threshold: float, emb_model) -> float:
    """
    Coverage 指标：用 gold meta-review 的【所有句子】作为关键点，
    计算生成文本覆盖了多少（余弦相似度 >= threshold 视为覆盖）。
    返回值：0~1，越大越好。
    """
    def split_sentences(text):
        sents = re.split(r'[.!?]\s+', text.strip())
        return [s.strip() for s in sents if len(s.strip()) >= 10]

    gold_pts = split_sentences(gold)
    gen_pts  = split_sentences(generated)

    if not gold_pts:
        return 1.0
    if not gen_pts:
        return 0.0

    E_gold = emb_model.encode(gold_pts, convert_to_tensor=True)
    E_gen  = emb_model.encode(gen_pts,  convert_to_tensor=True)
    sim_matrix = st_util.cos_sim(E_gold, E_gen).cpu().numpy()

    matched = 0
    used_gen = set()
    for j in range(len(gold_pts)):
        best_sim, best_t = -1.0, -1
        for t in range(len(gen_pts)):
            if t in used_gen:
                continue
            s = float(sim_matrix[j, t])
            if s > best_sim:
                best_sim, best_t = s, t
        if best_sim >= threshold and best_t >= 0:
            matched += 1
            used_gen.add(best_t)
    return matched / len(gold_pts)


cov_scores = []
supp_list, unsupp_list, contrad_list = [], [], []

print('⏳ 逐篇计算 Coverage & Hallucination...')
for i in tqdm(range(len(rows))):
    pid  = paper_ids[i]
    gen  = pred_reviews[i]
    gold = gold_reviews[i]
    revs = reviews_per_paper.get(pid, [gen])

    cov = coverage_all_sentences(gen, gold, threshold=THRESHOLD, emb_model=emb_model)
    cov_scores.append(cov)

    s, u, c = hallucination_rate(
        gen, revs, threshold=THRESHOLD,
        emb_model=emb_model,
        nli_tokenizer=nli_tokenizer,
        nli_model=nli_model,
    )
    supp_list.append(s)
    unsupp_list.append(u)
    contrad_list.append(c)

coverage_metrics = {'coverage_all': float(np.mean(cov_scores))}
hallucination_metrics = {
    'hallucination_supported_rate':    float(np.mean(supp_list)),
    'hallucination_unsupported_rate':  float(np.mean(unsupp_list)),
    'hallucination_contradicted_rate': float(np.mean(contrad_list)),
}

print('\n📊 Coverage（所有句子，越大越好）：')
for k, v in coverage_metrics.items():
    print(f'  {k}: {v:.4f}')

print('\n📊 Hallucination：')
for k, v in hallucination_metrics.items():
    print(f'  {k}: {v:.4f}')

## Cell 9：Decision-Review Consistency

In [ ]:
from evaluation import decision_review_consistency_rate

consistency = decision_review_consistency_rate(pred_reviews, pred_decisions)
print(f'📊 Decision-Review Consistency: {consistency:.4f}')

## Cell 10：汇总所有结果并保存

In [ ]:
# 如果跳过了 Cell 8，这里用空字典代替
try:
    _cov = coverage_metrics
    _hal = hallucination_metrics
except NameError:
    _cov = {}
    _hal = {}

try:
    _berts = bertscore_metrics
except NameError:
    _berts = {}

results = {
    **decision_metrics,
    **rouge_metrics,
    **_berts,
    **_cov,
    **_hal,
    'decision_review_consistency': consistency,
}

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f'✅ 结果已保存到: {OUT_PATH}')
print()
print(json.dumps(results, indent=2))